# MOD-06 · NB-03 — Difference-in-Differences & Event Studies
### Health Analytics with Python · Module 06: Causal Inference in Health Research
---
**Learning objectives**
- Understand the parallel trends assumption and when DiD is valid
- Implement classic 2×2 DiD and extend to panel data
- Test pre-treatment parallel trends with event study plots
- Apply Callaway-Sant'Anna staggered adoption DiD
- Handle heterogeneous treatment effects across cohorts
- Test the parallel trends assumption with placebo regressions

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `numpy`, `pandas`, `statsmodels`, `matplotlib`


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
print("Setup complete.")


## 2. Classic 2×2 DiD — Medicaid Expansion Example

**Scenario:** State Medicaid expansion under ACA (2014). Some states expanded (treated), others did not (control). Outcome: uninsured rate per county.

**DiD Estimator:**
```
DiD = (Y_treat_post - Y_treat_pre) - (Y_ctrl_post - Y_ctrl_pre)
```

**Assumption:** *Parallel trends* — absent the policy, treated and control states would have followed the same trend in the outcome.


In [ ]:
# Simulate Medicaid expansion data (state-level panel, 2012-2017)
np.random.seed(42)
N_states     = 50
T_periods    = 6      # 2012-2017
POLICY_YEAR  = 2014   # ACA Medicaid expansion
years        = list(range(2012, 2018))

# 25 states expand, 25 don't
expanded = np.array([1]*25 + [0]*25)
state_fe = np.random.normal(0, 2, N_states)  # state fixed effects
# True ATT: expansion reduces uninsured rate by 4 percentage points
TRUE_ATT = -4.0

records = []
for s in range(N_states):
    for i, yr in enumerate(years):
        post   = int(yr >= POLICY_YEAR)
        treat  = expanded[s]
        # Pre-trend (same slope for both)
        trend  = -0.8 * (yr - 2012)  # national trend: uninsured falling
        # Treatment effect only for treated states after policy
        effect = TRUE_ATT * treat * post
        # Base uninsured rate + state FE + trend + effect + noise
        uninsured = (18 + state_fe[s] + trend + effect + 3*treat
                     + np.random.normal(0, 1.5))
        records.append({"state":s,"year":yr,"expanded":treat,
                        "post":post,"uninsured_pct":uninsured})

df_did = pd.DataFrame(records)
print(f"DiD panel: {len(df_did)} state-years | {N_states} states | {T_periods} years")
print(f"Expansion states: {expanded.sum()} | Non-expansion: {(1-expanded).sum()}")

# 2x2 DiD table
dd_table = df_did.groupby(["expanded","post"])["uninsured_pct"].mean().unstack()
dd_table.index = ["Non-expansion","Expansion"]
dd_table.columns = ["Pre-policy (2012-13)","Post-policy (2014-17)"]
print("
2x2 DiD Table:")
display(dd_table.round(2))
DiD_est = (dd_table.iloc[1,1]-dd_table.iloc[1,0]) - (dd_table.iloc[0,1]-dd_table.iloc[0,0])
print(f"
DiD estimate: {DiD_est:.3f} pp | True ATT: {TRUE_ATT:.1f} pp")


## 3. Regression DiD with Fixed Effects

In [ ]:
# Regression DiD: Y = alpha + beta1*Treat + beta2*Post + beta3*(Treat*Post) + FE + e
# beta3 is the DiD estimator

# Two-Way Fixed Effects (TWFE) model
df_did["state_year"] = df_did["state"].astype(str) + "_" + df_did["year"].astype(str)
model_twfe = smf.ols("uninsured_pct ~ expanded * C(post) + C(state) + C(year)",
                      data=df_did).fit(cov_type="cluster", cov_kwds={"groups":df_did["state"]})

did_coef = model_twfe.params["expanded:C(post)[T.1]"]
did_ci   = model_twfe.conf_int().loc["expanded:C(post)[T.1]"]
did_se   = model_twfe.bse["expanded:C(post)[T.1]"]

print("Two-Way Fixed Effects DiD:")
print(f"  DiD estimate   : {did_coef:.3f} pp")
print(f"  SE (clustered) : {did_se:.3f}")
print(f"  95% CI         : ({did_ci[0]:.3f}, {did_ci[1]:.3f})")
print(f"  True ATT       : {TRUE_ATT:.1f} pp")
print(f"  Bias           : {did_coef - TRUE_ATT:+.3f} pp")

# Visualise parallel trends
fig, axes = plt.subplots(1,2,figsize=(14,5))
agg = df_did.groupby(["year","expanded"])["uninsured_pct"].mean().unstack()
agg.columns = ["Non-expansion","Expansion"]
agg.plot(ax=axes[0], color=["#4878CF","#D65F5F"], marker="o", lw=2)
axes[0].axvline(POLICY_YEAR-0.5, color="black", ls="--", lw=1.5, label="ACA Expansion (2014)")
axes[0].set_ylabel("Uninsured rate (%)"); axes[0].set_xlabel("Year")
axes[0].set_title("DiD: Parallel Trends Plot
(both groups track together pre-policy)")
axes[0].legend(fontsize=9)

# Counterfactual
post_trend_ctrl = agg.loc[years[2]:,"Non-expansion"].diff().mean()
counterfactual = pd.Series({yr: agg.loc[2013,"Expansion"] + post_trend_ctrl*(yr-2013)
                             for yr in years[2:]})
axes[0].plot(counterfactual.index, counterfactual.values, "r--", lw=2, label="Counterfactual (no expansion)")
axes[0].legend(fontsize=8)

# Effect size by year (event study)
agg["diff"] = agg["Expansion"] - agg["Non-expansion"]
axes[1].bar(agg.index, agg["diff"], color=["#AEC6CF" if yr<POLICY_YEAR else "#D65F5F" for yr in agg.index], alpha=0.85)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].axvline(POLICY_YEAR-0.5, color="black", ls="--", lw=1.5, label="Policy year")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Treated - Control gap (pp)")
axes[1].set_title("Pre-trends check + Treatment effect by year")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/did_parallel_trends.png",bbox_inches="tight"); plt.show()


## 4. Event Study Regression (Dynamic DiD)

In [ ]:
# Event study: estimate treatment effect at each time relative to policy
# Yi,t = alpha_i + lambda_t + sum_k(beta_k * D_i * 1[t=k]) + e
# k = time relative to treatment (k=-1 is reference period)

df_did["rel_year"] = df_did["year"] - POLICY_YEAR
df_did["rel_cat"]  = df_did["rel_year"].clip(-3, 3)  # bin at edges
# Create interaction terms for each relative year
for k in range(-3, 4):
    if k != -1:  # omit k=-1 as reference
        df_did[f"D_k{k}"] = (df_did["expanded"] * (df_did["rel_cat"]==k)).astype(int)

# Build formula
interaction_terms = [f"D_k{k}" for k in range(-3, 4) if k != -1]
formula = "uninsured_pct ~ " + " + ".join(interaction_terms) + " + C(state) + C(year)"
model_es = smf.ols(formula, data=df_did).fit(cov_type="cluster",
                                               cov_kwds={"groups":df_did["state"]})

# Extract event study coefficients
event_coefs = {}
for k in range(-3, 4):
    if k == -1:
        event_coefs[k] = {"coef":0, "ci_lo":0, "ci_hi":0}
    else:
        param = f"D_k{k}"
        coef  = model_es.params[param]
        ci    = model_es.conf_int().loc[param]
        event_coefs[k] = {"coef":coef, "ci_lo":ci[0], "ci_hi":ci[1]}

ks   = sorted(event_coefs.keys())
coefs = [event_coefs[k]["coef"] for k in ks]
ci_lo = [event_coefs[k]["ci_lo"] for k in ks]
ci_hi = [event_coefs[k]["ci_hi"] for k in ks]

fig, ax = plt.subplots(figsize=(10,5))
ax.plot(ks, coefs, "o-", color="#D65F5F", lw=2, ms=8, zorder=3)
ax.fill_between(ks, ci_lo, ci_hi, alpha=0.2, color="#D65F5F")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.axvline(-0.5, color="gray", ls="--", lw=1.2, label="Policy implementation")
ax.set_xlabel("Years relative to Medicaid expansion"); ax.set_ylabel("Effect on uninsured rate (pp)")
ax.set_title("Event Study: Dynamic Treatment Effects
(pre-treatment coefficients should be ~0 if parallel trends hold)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/event_study.png",bbox_inches="tight"); plt.show()

pre_trend_test = [(k, event_coefs[k]["coef"]) for k in ks if k < -1]
print("Pre-treatment event study coefficients (should be near 0):")
for k, coef in pre_trend_test:
    print(f"  k={k}: {coef:.3f}  {'OK' if abs(coef)<1 else 'CONCERNING'}")


## 5. Staggered Adoption — Callaway-Sant'Anna

In [ ]:
# When treatment rolls out at different times, naive TWFE gives
# a weighted average that can be negative even when all cohort ATTs are positive
# Callaway-Sant'Anna (2021) estimates cohort-specific ATTs

np.random.seed(42)
N_states2  = 60
cohorts    = [2014,2015,2016,None]  # None = never-treated
cohort_assign = np.repeat(cohorts, [15,15,15,15])
TRUE_COHORT_ATT = {2014:-5.0, 2015:-4.0, 2016:-3.0}  # heterogeneous effects

records2 = []
state_fe2 = np.random.normal(0,2,N_states2)
for s in range(N_states2):
    cohort = cohort_assign[s]
    for yr in range(2012, 2020):
        post   = (cohort is not None and yr >= cohort)
        effect = TRUE_COHORT_ATT.get(cohort, 0) if post else 0
        unins  = (18 + state_fe2[s] - 0.8*(yr-2012) + effect
                  + np.random.normal(0,1.5))
        records2.append({"state":s,"year":yr,"cohort":str(cohort),
                         "treated":int(cohort is not None),"post":int(post),
                         "uninsured_pct":unins})

df_staggered = pd.DataFrame(records2)

# Simple cohort-specific DiD (Callaway-Sant'Anna principle)
# Compare each treated cohort to never-treated states in the relevant period
never_treated = df_staggered[df_staggered.cohort=="None"]
cs_results = []
for g in ["2014","2015","2016"]:
    g_int = int(g)
    cohort_data = df_staggered[df_staggered.cohort==g]
    for t in range(g_int, 2020):
        # ATT(g, t) using never-treated as comparison
        mu_g_t = cohort_data[cohort_data.year==t]["uninsured_pct"].mean()
        mu_g_base = cohort_data[cohort_data.year==g_int-1]["uninsured_pct"].mean()
        mu_nt_t = never_treated[never_treated.year==t]["uninsured_pct"].mean()
        mu_nt_base = never_treated[never_treated.year==g_int-1]["uninsured_pct"].mean()
        att_gt = (mu_g_t - mu_g_base) - (mu_nt_t - mu_nt_base)
        cs_results.append({"cohort":g,"year":t,"att":att_gt,
                            "true_att":TRUE_COHORT_ATT[g_int]})

df_cs = pd.DataFrame(cs_results)

fig, ax = plt.subplots(figsize=(11,5))
colors_cohort = {"2014":"#D65F5F","2015":"#4878CF","2016":"#6ACC65"}
for g in ["2014","2015","2016"]:
    sub = df_cs[df_cs.cohort==g]
    ax.plot(sub.year, sub.att, "o-", color=colors_cohort[g], lw=2, ms=7, label=f"Cohort {g}")
    ax.axhline(TRUE_COHORT_ATT[int(g)], color=colors_cohort[g], ls=":", lw=1.5, alpha=0.7)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Calendar year"); ax.set_ylabel("ATT(g,t) — pp reduction in uninsured rate")
ax.set_title("Callaway-Sant'Anna: Cohort-specific ATTs
(dotted = true effect for each cohort)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/cs_did.png",bbox_inches="tight"); plt.show()

# Aggregate ATT
agg_att = df_cs.groupby("cohort")["att"].mean()
print("Callaway-Sant'Anna Aggregate ATTs:")
for g in ["2014","2015","2016"]:
    print(f"  Cohort {g}: Est={agg_att[g]:.2f} pp | True={TRUE_COHORT_ATT[int(g)]:.1f} pp")


## 6. Knowledge Check
1. The event study shows a pre-trend coefficient of -1.2 pp at k=-2. What does this imply for the parallel trends assumption?
2. Why does naive TWFE DiD fail with staggered adoption and heterogeneous effects?
3. A colleague proposes using the "always-treated" states as controls for never-treated states. What problem does this create?
4. Your DiD estimate is -4.2 pp but the 95% CI is (-8.1, -0.3). Is this statistically significant? What drives the wide CI?
5. Implement a placebo test: re-run the DiD using 2012 as the fake policy year. What should the placebo estimate be?


In [ ]:
# Exercise 5 — Placebo DiD test
df_placebo = df_did.copy()
df_placebo["post_placebo"] = (df_placebo["year"] >= 2013).astype(int)  # fake 2013 cutoff
df_placebo["treat_post_placebo"] = df_placebo["expanded"] * df_placebo["post_placebo"]
# Restrict to pre-policy data only
df_pre = df_placebo[df_placebo.year < POLICY_YEAR].copy()

model_placebo = smf.ols("uninsured_pct ~ expanded*post_placebo + C(state) + C(year)",
                         data=df_pre).fit(cov_type="cluster", cov_kwds={"groups":df_pre["state"]})
placebo_coef = model_placebo.params["expanded:post_placebo"]
placebo_ci   = model_placebo.conf_int().loc["expanded:post_placebo"]

print("Placebo DiD Test (fake policy year = 2013, pre-period data only):")
print(f"  Placebo estimate: {placebo_coef:.3f} pp")
print(f"  95% CI: ({placebo_ci[0]:.3f}, {placebo_ci[1]:.3f})")
print(f"  p-value: {model_placebo.pvalues['expanded:post_placebo']:.4f}")
passes = abs(placebo_coef) < 1.5 and placebo_ci[0] < 0 < placebo_ci[1]
print(f"  Parallel trends test: {'PASS (no pre-trend)' if passes else 'FAIL (pre-trend detected)'}")


---
**Next -> NB-04: Interrupted Time Series (ITS) & CausalImpact**